# NumPy만으로 MNIST 98% 달성하기

**30일 딥러닝 완전 정복 · Day 14 · 2주차 프로젝트** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- MNIST 6만 장을 불러와 학습/검증/테스트로 나눈다 (torchvision은 다운로드에만 사용)
- 2주 동안 만든 부품(Linear, ReLU, Dropout, Softmax-CE, AdamW, 조기 종료)을 조립한다
- 784-256-128-10 MLP로 테스트 정확도 98% 이상을 목표로 학습한다
- 학습 곡선·틀린 예시·혼동 행렬로 결과를 분석한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

## 1. 데이터 준비
학습 55,000 / 검증 5,000 / 테스트 10,000으로 나눕니다. 정규화 통계는 **학습 세트로만** 계산합니다(Day 14 데이터 누수).

In [ ]:
import numpy as np, torch
from torchvision import datasets
mn_tr = datasets.MNIST("data", train=True, download=True)
mn_te = datasets.MNIST("data", train=False, download=True)
X_all = mn_tr.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_all = mn_tr.targets.numpy()
X_test = mn_te.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = mn_te.targets.numpy()
rng = np.random.default_rng(0)
perm = rng.permutation(len(y_all))
tr_idx, va_idx = perm[:55000], perm[55000:]
X_tr, y_tr, X_va, y_va = X_all[tr_idx], y_all[tr_idx], X_all[va_idx], y_all[va_idx]
mu, sd = X_tr.mean(), X_tr.std()                      # 학습 세트 통계로만 정규화
X_tr, X_va, X_test = (X_tr - mu) / sd, (X_va - mu) / sd, (X_test - mu) / sd
print(X_tr.shape, X_va.shape, X_test.shape)

## 2. 층 구현 (Day 11, 13, 14)
모든 층은 `forward`, `backward`, `params`를 가집니다. `params`는 `(파라미터, 기울기, 가중치감쇠 적용여부)`를 돌려줍니다.

In [ ]:
class Linear:
    def __init__(self, d_in, d_out):
        self.W = (rng.standard_normal((d_in, d_out)) * np.sqrt(2.0 / d_in)).astype(np.float32)   # He 초기화
        self.b = np.zeros(d_out, np.float32)
        self.dW = np.zeros_like(self.W); self.db = np.zeros_like(self.b)
    def forward(self, x): self.x = x; return x @ self.W + self.b
    def backward(self, g):
        self.dW[...] = self.x.T @ g; self.db[...] = g.sum(0)     # 제자리 대입: params()가 같은 배열을 가리키게
        return g @ self.W.T
    def params(self): return [(self.W, self.dW, True), (self.b, self.db, False)]

class ReLU:
    def forward(self, x): self.m = x > 0; return x * self.m
    def backward(self, g): return g * self.m
    def params(self): return []

class Dropout:
    def __init__(self, p): self.p, self.training = p, True
    def forward(self, x):
        if not self.training or self.p == 0: return x
        self.mask = (rng.random(x.shape) >= self.p).astype(np.float32) / (1 - self.p)
        return x * self.mask
    def backward(self, g): return g * self.mask if (self.training and self.p > 0) else g
    def params(self): return []

class Model:
    def __init__(self, *layers): self.layers = list(layers)
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def backward(self, g):
        for l in reversed(self.layers): g = l.backward(g)
    def params(self): return [p for l in self.layers for p in l.params()]
    def train(self, mode=True):
        for l in self.layers:
            if hasattr(l, "training"): l.training = mode

def softmax_ce(logits, y):
    z = logits - logits.max(1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(1, keepdims=True))
    P = np.exp(logp); g = P.copy(); g[np.arange(len(y)), y] -= 1
    return -logp[np.arange(len(y)), y].mean(), g / len(y)

class AdamW:
    def __init__(self, model, lr=1e-3, wd=1e-4, b1=0.9, b2=0.999, eps=1e-8):
        self.model, self.lr, self.wd, self.b1, self.b2, self.eps, self.t = model, lr, wd, b1, b2, eps, 0
        self.m = [np.zeros_like(p) for p, _, _ in model.params()]
        self.v = [np.zeros_like(p) for p, _, _ in model.params()]
    def step(self):
        self.t += 1
        for (p, g, decay), m, v in zip(self.model.params(), self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            mh = m / (1 - self.b1 ** self.t); vh = v / (1 - self.b2 ** self.t)
            p -= self.lr * mh / (np.sqrt(vh) + self.eps)
            if decay: p -= self.lr * self.wd * p

## 3. 기울기 검증
새 부품을 쓰기 전에 반드시 통과시킵니다 (Day 11).

In [ ]:
def grad_check(model, X, y, n=10, h=1e-3):
    model.train(False)
    X = X.astype(np.float64)
    for l in model.layers:                       # 정밀도를 위해 float64로
        if isinstance(l, Linear): l.W = l.W.astype(np.float64); l.b = l.b.astype(np.float64); l.dW = np.zeros_like(l.W); l.db = np.zeros_like(l.b)
    _, g = softmax_ce(model.forward(X), y); model.backward(g)
    worst = 0
    for p, dp, _ in model.params():
        for _ in range(n):
            i = tuple(rng.integers(0, s) for s in p.shape); old = p[i]
            p[i] = old + h; lp, _ = softmax_ce(model.forward(X), y)
            p[i] = old - h; lm, _ = softmax_ce(model.forward(X), y); p[i] = old
            num = (lp - lm) / (2 * h); worst = max(worst, abs(num - dp[i]) / max(abs(num), abs(dp[i]), 1e-8))
    return worst

small = Model(Linear(784, 16), ReLU(), Linear(16, 10))
print("max relative error:", grad_check(small, X_tr[:8], y_tr[:8]))

## 4. 학습 (조기 종료 포함)
CPU에서도 에폭당 수 초~십수 초면 돕니다. 이 프로젝트는 GPU가 필요 없습니다.

In [ ]:
import time
model = Model(Linear(784, 256), ReLU(), Dropout(0.2), Linear(256, 128), ReLU(), Dropout(0.2), Linear(128, 10))
opt = AdamW(model, lr=1e-3, wd=1e-4)

def evaluate(X, y, bs=2000):
    model.train(False); losses, correct = [], 0
    for s in range(0, len(y), bs):
        l, _ = softmax_ce(out := model.forward(X[s:s+bs]), y[s:s+bs]); losses.append(l * len(out))
        correct += (out.argmax(1) == y[s:s+bs]).sum()
    return sum(losses) / len(y), correct / len(y)

hist = {"tr_loss": [], "va_loss": [], "va_acc": []}
best, best_params, patience, bad = 0, None, 5, 0
B = 128
for epoch in range(40):
    t0 = time.time(); model.train(True); idx = rng.permutation(len(y_tr)); run = 0
    for s in range(0, len(idx), B):
        bi = idx[s:s+B]
        loss, g = softmax_ce(model.forward(X_tr[bi]), y_tr[bi]); model.backward(g); opt.step(); run += loss * len(bi)
    va_loss, va_acc = evaluate(X_va, y_va)
    hist["tr_loss"].append(run / len(idx)); hist["va_loss"].append(va_loss); hist["va_acc"].append(va_acc)
    print(f"epoch {epoch:2d}  train loss {run/len(idx):.4f}  val loss {va_loss:.4f}  val acc {va_acc:.4f}  ({time.time()-t0:.1f}s)")
    if va_acc > best: best, bad, best_params = va_acc, 0, [p.copy() for p, _, _ in model.params()]
    else:
        bad += 1
        if bad >= patience: print("early stopping"); break
for (p, _, _), bp in zip(model.params(), best_params): p[...] = bp
print("best val acc:", best)

## 5. 최종 테스트 (딱 한 번!)

In [ ]:
test_loss, test_acc = evaluate(X_test, y_test)
print(f"TEST accuracy: {test_acc:.4f}   {'🎉 목표 달성!' if test_acc >= 0.98 else '조금 더 튜닝해 보세요 (은닉 크기, 드롭아웃, 학습률)'}")

## 6. 분석

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(hist["va_acc"]); ax[1].set_title("val accuracy"); plt.show()

model.train(False)
pred = model.forward(X_test).argmax(1)
wrong = np.where(pred != y_test)[0][:16]
fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for a, i in zip(axs.ravel(), wrong):
    a.imshow((X_test[i] * sd + mu).reshape(28, 28), cmap="gray"); a.set_title(f"true {y_test[i]} / pred {pred[i]}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

cm = np.zeros((10, 10), int)
for t, p in zip(y_test, pred): cm[t, p] += 1
plt.figure(figsize=(6, 5)); plt.imshow(np.log1p(cm), cmap="Blues"); plt.colorbar(label="log(1+count)")
plt.xlabel("predicted"); plt.ylabel("true"); plt.xticks(range(10)); plt.yticks(range(10)); plt.show()
print("per-class accuracy:", (cm.diagonal() / cm.sum(1)).round(3))

## 7. 도전 과제
1. 은닉층을 512-256으로 키우고 드롭아웃을 0.3으로 올려 보세요. 98.5%를 넘길 수 있나요?
2. Day 13의 BatchNorm을 구현해 Linear와 ReLU 사이에 넣어 보세요 (기울기 검증 필수!).
3. 학습률 스케줄(에폭마다 0.9배)을 추가해 보세요.

> 이 노트북의 모든 코드는 여러분이 2주 동안 직접 유도한 것입니다. 다음 주부터는 PyTorch가 이 일을 대신합니다.